# S4 — 特徵工程思維：從觀察到創造（Solution）

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/titanic/train.csv`（講授）、`datasets/spaceship-titanic/train.csv`（練習）  
> **學完能幹嘛**：從原始欄位中創造出有預測力的新特徵

---

**一句話口訣：原始欄位是食材，特徵工程是料理 — 同一塊豆腐，紅燒和涼拌完全不同價值。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/titanic/train.csv')

---
## 核心觀念 1／3 — 組合特徵

In [ ]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x='FamilySize', y='Survived', data=df, ax=axes[0])
axes[0].set_title('存活率 by FamilySize')
sns.barplot(x='IsAlone', y='Survived', data=df, ax=axes[1], palette=['#66b2ff', '#ff9999'])
axes[1].set_title('存活率：獨行 vs 有同伴')
axes[1].set_xticklabels(['有同伴', '獨行'])
plt.tight_layout()
plt.show()

---
## 核心觀念 2／3 — 文字拆解

In [ ]:
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < 10].index
df['Title'] = df['Title'].replace(rare_titles, 'Rare')

fig, ax = plt.subplots(figsize=(8, 5))
df.groupby('Title')['Survived'].mean().sort_values(ascending=False).plot(
    kind='bar', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('存活率 by Title')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
df['Deck'] = df['Cabin'].str[0]

fig, ax = plt.subplots(figsize=(8, 5))
df.groupby('Deck')['Survived'].mean().sort_values(ascending=False).plot(
    kind='bar', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('存活率 by Deck')
plt.tight_layout()
plt.show()

---
## 核心觀念 3／3 — 分箱與交互特徵

In [ ]:
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
order = ['female_1', 'female_2', 'female_3', 'male_1', 'male_2', 'male_3']
sns.barplot(x='Sex_Pclass', y='Survived', data=df, ax=ax, order=order)
ax.set_title('存活率 by Sex × Pclass')
plt.tight_layout()
plt.show()

print(pd.pivot_table(df, values='Survived', index='Sex', columns='Pclass', aggfunc='mean').round(3))

---
## 課堂練習 — Solution

### 🟢 送分題

In [ ]:
# 🟢 送分題 Solution
sp = pd.read_csv('datasets/spaceship-titanic/train.csv')
sp.info()

In [ ]:
# TotalSpending
spend_cols = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
sp['TotalSpending'] = sp[spend_cols].sum(axis=1)

# 需要把 Transported 轉成數值
sp['Transported_int'] = sp['Transported'].astype(int)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x='Transported', y='TotalSpending', data=sp, ax=ax)
ax.set_title('平均 TotalSpending by Transported', fontsize=14)
plt.tight_layout()
plt.show()

print('Transported 的平均消費：')
print(sp.groupby('Transported')['TotalSpending'].mean().round(1))
print('\n→ 被傳送的人消費明顯較低 — TotalSpending 是好特徵')

### 🟡 核心題

In [ ]:
# 🟡 核心題 Solution

# 1. 拆解 Cabin
cabin_split = sp['Cabin'].str.split('/', expand=True)
sp['Cabin_deck'] = cabin_split[0]
sp['Cabin_num'] = cabin_split[1]
sp['Cabin_side'] = cabin_split[2]

print('Cabin 拆解結果：')
print(sp[['Cabin', 'Cabin_deck', 'Cabin_num', 'Cabin_side']].head())

In [ ]:
# 2. Transported rate by Cabin_deck
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

deck_rate = sp.groupby('Cabin_deck')['Transported_int'].mean().sort_values(ascending=False)
deck_rate.plot(kind='bar', color='steelblue', edgecolor='white', ax=axes[0])
axes[0].set_title('Transported Rate by Cabin_deck', fontsize=12)
axes[0].set_ylabel('Transported Rate')

# 3. Transported rate by Cabin_side
side_rate = sp.groupby('Cabin_side')['Transported_int'].mean()
side_rate.plot(kind='bar', color='coral', edgecolor='white', ax=axes[1])
axes[1].set_title('Transported Rate by Cabin_side (P/S)', fontsize=12)
axes[1].set_ylabel('Transported Rate')

plt.tight_layout()
plt.show()

print('→ Deck B, C 的 transported rate 較高')
print('→ S (Starboard) 略高於 P (Port) — 船的左右側影響不大')

### 🔴 挑戰題

In [ ]:
# 🔴 挑戰題 Solution

# 1. IsVIP
sp['IsVIP'] = sp['VIP'].fillna(False).astype(int)

# 2. SpendingCategory（處理 TotalSpending 中大量的 0）
# 很多人消費為 0，qcut 會失敗，改用 cut 或特殊處理
sp['SpendingCategory'] = pd.cut(
    sp['TotalSpending'],
    bins=[-1, 0, 500, sp['TotalSpending'].max()],
    labels=['Zero', 'Low-Mid', 'High']
)

# 3. Deck_Side 交互特徵
sp['Deck_Side'] = sp['Cabin_deck'].fillna('Unknown') + '_' + sp['Cabin_side'].fillna('Unknown')

# 4. 2×2 子圖
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# IsVIP
sns.barplot(x='IsVIP', y='Transported_int', data=sp, ax=axes[0, 0])
axes[0, 0].set_title('Transported by IsVIP')

# SpendingCategory
sns.barplot(x='SpendingCategory', y='Transported_int', data=sp, ax=axes[0, 1])
axes[0, 1].set_title('Transported by SpendingCategory')

# Cabin_deck
sns.barplot(x='Cabin_deck', y='Transported_int', data=sp, ax=axes[1, 0])
axes[1, 0].set_title('Transported by Cabin_deck')

# Cabin_side
sns.barplot(x='Cabin_side', y='Transported_int', data=sp, ax=axes[1, 1])
axes[1, 1].set_title('Transported by Cabin_side')

plt.tight_layout()
plt.show()

# 5. 區分度比較
print('各特徵的區分度（Transported rate 的標準差）：')
for feat in ['IsVIP', 'SpendingCategory', 'Cabin_deck', 'Cabin_side']:
    rates = sp.groupby(feat)['Transported_int'].mean()
    print(f'  {feat:20s} → std = {rates.std():.4f} (range: {rates.min():.3f} ~ {rates.max():.3f})')

print('\n→ SpendingCategory 和 Cabin_deck 的區分度最高')

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|:---------|:-------|
| 欄位相加 | `df['new'] = df['a'] + df['b']` |
| 二值化 | `df['flag'] = (df['col'] == val).astype(int)` |
| 正則拆解 | `df['col'].str.extract(r'(pattern)')` |
| 字串分割 | `df['col'].str.split('/').str[0]` |
| 等頻分箱 | `pd.qcut(df['col'], q=4, labels=[...])` |
| 自訂分箱 | `pd.cut(df['col'], bins=[...], labels=[...])` |
| 交互特徵 | `df['a'] + '_' + df['b'].astype(str)` |
| 快速驗證 | `df.groupby('new_feat')['target'].mean()` |

**下節預告 S5**：EDA Capstone — 從零到報告的完整流程